#### Build to Google Drive <font color="DarkSeaGreen">/ GROMACS-on-Colab</font> [<img src="https://github.githubassets.com/favicons/favicon.png" width="16">](https://github.com/bioinfkaustin/gromacs-on-colab)
<small>Suite: `Build_to_Google_Drive.ipynb` | [`GROMACS_for_CHARMM-GUI.ipynb`](https://colab.research.google.com/github/bioinfkaustin/gromacs-on-colab/blob/main/GROMACS_for_CHARMM-GUI.ipynb) | [`GROMACS_for_production.ipynb`](https://colab.research.google.com/github/bioinfkaustin/gromacs-on-colab/blob/main/GROMACS_for_production.ipynb) | [`Trajectory_analysis_tools.ipynb`](https://colab.research.google.com/github/bioinfkaustin/gromacs-on-colab/blob/main/Trajectory_analysis_tools.ipynb)</small>

#### Documentation
**Before using this notebook, please click the *↳ cells hidden* button below to show the documentation.**

##### License

> This notebook as a work of software is licensed under the terms of the [AGPL-3.0](https://opensource.org/licenses/AGPL-3.0) or later.

##### About this software

> This notebook compiles and installs **GROMACS** and other libraries and utilities for running molecular dynamics simulations. It then caches the installations in your connected **Google Drive**, for later use in other notebooks.
>
> <font color="maroon">$\lower{0.25ex}{\smash{\LARGE ⚠}}$ Recommended runtime: **L4 GPU** with **High-RAM** (or any GPU runtime). The build needs the CUDA toolkit. Compilation from source takes ~15-30 minutes on L4.</font>

#### Installation

**Run this installation by clicking *Runtime -> Run all* in the toolbar.**

In [ ]:
#@markdown In the following cells, applications will be downloaded from the internet and compiled and/or installed to a **persistent cache** in your Google Drive.
#@markdown
#@markdown This cell sets up the cache folder.

import os

if not os.path.isdir("/content/drive/MyDrive"):
  from google.colab import drive
  drive.mount("/content/drive")

if not os.path.isdir("/content/drive/MyDrive"):
  raise RuntimeError("Error: could not connect to Google Drive")

storage = "/content/drive/MyDrive/gromacs-on-colab"
os.makedirs(storage, exist_ok=True)
%env STORAGE={storage}

if "START" not in os.environ or not os.environ["START"]:
  %env START={os.getcwd()}
else:
  %cd {os.environ["START"]}

In [ ]:
%%bash
#@markdown **GROMACS** is downloaded prebuilt from [this notebook's GitHub repository](https://github.com/bioinfkaustin/gromacs-on-colab).
#@markdown
#@markdown If not available, it is instead compiled from source code. (This takes ~15-30 minutes on an L4 GPU runtime.)

if [[ -d "/usr/local/gromacs" ]]; then
  exit 0  # already installed
fi

gromacs_vers="2026.1" #@param {type: "string"}
cache_gromacs="${STORAGE}/gromacs-${gromacs_vers}.tar.gz"

if [[ -s "${cache_gromacs}" ]]; then
  tar -xzf "${cache_gromacs}" -C "/usr/local"
else
  # Try to get a prebuilt archive...
  wget -q "https://raw.githubusercontent.com/bioinfkaustin/gromacs-on-colab/main/prebuilt/gromacs-${gromacs_vers}.tar.gz"
  if [[ -s "gromacs-${gromacs_vers}.tar.gz" ]]; then
    tar -xzf "gromacs-${gromacs_vers}.tar.gz" -C "/usr/local"

    # Cache
    cp "gromacs-${gromacs_vers}.tar.gz" "${cache_gromacs}"

  # Prebuilt archive not available, so download the source code and build it...
  else
    wget -q "https://ftp.gromacs.org/gromacs/gromacs-${gromacs_vers}.tar.gz"
    if [[ ! -s "gromacs-${gromacs_vers}.tar.gz" ]]; then
      echo "Error: could not download: gromacs-${gromacs_vers}.tar.gz" >&2
      exit 1
    fi
    tar -xzf "gromacs-${gromacs_vers}.tar.gz"
    rm "gromacs-${gromacs_vers}.tar.gz"

    cd "gromacs-${gromacs_vers}"
    mkdir "build"
    cd "build"
    # Compute capabilities cover Colab GPUs: T4(75), A100(80), RTX30(86), L4(89), H100(90).
    # PTX for sm_90 enables JIT fallback to newer architectures.
    cmake .. \
      -DCMAKE_BUILD_TYPE=Release \
      -DGMX_BUILD_OWN_FFTW=ON \
      -DGMX_GPU=CUDA \
      -DGMX_CUDA_TARGET_SM="75;80;86;89;90" \
      -DGMX_CUDA_TARGET_COMPUTE="90" \
      -DGMX_SIMD=AVX2_256
    make -j $(nproc)
    make install # -> /usr/local/gromacs

    # Cache
    tar -czf "my_gromacs.tar.gz" -C "/usr/local" "gromacs"
    mv "my_gromacs.tar.gz" "${cache_gromacs}"
  fi
fi

In [ ]:
%%bash
#@markdown Install Python dependencies into the Colab system Python (3.12+):
#@markdown - **Open Babel** CLI (`obabel`) via apt, used for ligand format conversion
#@markdown - **Biopython**, **NetworkX**, **NumPy** via pip, used by `superpose.py` and `cgenff_charmm2gmx.py`
#@markdown
#@markdown These install in seconds; no Drive cache is used.

set -e

if ! command -v obabel >/dev/null; then
  apt-get install -qq -y openbabel >/dev/null
fi

pip install --quiet biopython networkx numpy

In [ ]:
%%bash
#@markdown The CHARMM36 forcefield is downloaded.

if [[ -d "${START}/charmm36.ff" ]]; then
  exit 0  # already installed
fi

charmm36_vers="feb2026_cgenff-5.0" #@param {type: "string"}
cache_charmm36="${STORAGE}/charmm36-${charmm36_vers}.tar.gz"

if [[ -s "${cache_charmm36}" ]]; then
  tar -xzf "${cache_charmm36}" -C "${START}"
else
  wget -q -O "charmm36-${charmm36_vers}.ff.tgz" "https://mackerell.umaryland.edu/download.php?filename=CHARMM_ff_params_files/charmm36-${charmm36_vers}.ff.tgz"
  if [[ ! -s "charmm36-${charmm36_vers}.ff.tgz" ]]; then
    echo "Error: could not download: charmm36-${charmm36_vers}.ff.tgz" >&2
    exit 1
  fi
  tar -xzf "charmm36-${charmm36_vers}.ff.tgz"
  rm "charmm36-${charmm36_vers}.ff.tgz"

  # The extracted top-level directory matches the archive name
  mv "charmm36-${charmm36_vers}.ff" "${START}/charmm36.ff"

  # Cache
  tar -czf "my_charmm36.tar.gz" -C "${START}" "charmm36.ff"
  mv "my_charmm36.tar.gz" "${cache_charmm36}"
fi

In [ ]:
%%bash
#@markdown The utility **`cgenff_charmm2gmx.py`** is downloaded from the [Lemkul-Lab repository](https://github.com/Lemkul-Lab/cgenff_charmm2gmx) at a pinned commit.
#@markdown The pinned commit (2026-04) is the first to support both CGenFF 4.6 and 5.0 concurrently, matching the `feb2026_cgenff-5.0` forcefield variant.

mkdir -p "${START}/bin"
target="${START}/bin/cgenff_charmm2gmx.py"

if [[ -x "${target}" ]]; then
  exit 0  # already installed
fi

cgenff_commit="f71070b0ce1c256754c486e3b0b6490a5ed481f8" #@param {type: "string"}
cache_cgenff="${STORAGE}/cgenff_charmm2gmx-${cgenff_commit:0:8}.py"

if [[ -s "${cache_cgenff}" ]]; then
  cp "${cache_cgenff}" "${target}"
else
  wget -q -O "${target}" "https://raw.githubusercontent.com/Lemkul-Lab/cgenff_charmm2gmx/${cgenff_commit}/cgenff_charmm2gmx.py"
  if [[ ! -s "${target}" ]]; then
    echo "Error: could not download cgenff_charmm2gmx.py at commit ${cgenff_commit}" >&2
    exit 1
  fi
  cp "${target}" "${cache_cgenff}"
fi

chmod +x "${target}"

In [ ]:
#@markdown Finally, disconnect the runtime.
disconnect = True #@param {type: "boolean"}

if disconnect:
  from google.colab import drive, runtime
  drive.flush_and_unmount()
  runtime.unassign()